# Proposal Distances vs Mean and Sigma

This notebook reads the proposal-moments CSV and computes Gaussian distance measures in-notebook:
- \($D_{KL}(q_{init}\|p(x\mid y))\)
- \($D_{KL}(p(x\mid y)\|q_{init})\)
- \($W_2(q_{init}, p(x\mid y))\)
- \(H(q_{init}, p(x\mid y))\)

It also reports prior-vs-posterior baselines for the same metrics.

In [ ]:
import copy
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd

plt.style.use("plots.style")

In [ ]:
# Update this path if needed
csv_path = Path(
    "/home/michael/git_repos/aspire-analyses/kl_criteria/out_test/proposal_moments_efficiency.csv"
)
csv_path_fix_mu = Path(
    "/home/michael/git_repos/aspire-analyses/kl_criteria/out_fix_mu/proposal_moments_efficiency.csv"
)
csv_path_fix_sigma = Path(
    "/home/michael/git_repos/aspire-analyses/kl_criteria/out_fix_sigma/proposal_moments_efficiency.csv"
)

df = pd.read_csv(csv_path)
print(f"Loaded {len(df)} rows from {csv_path}")
df.head()

df_fix_mu = pd.read_csv(csv_path_fix_mu)
print(f"Loaded {len(df_fix_mu)} rows from {csv_path_fix_mu}")
df_fix_mu.head()

df_fix_sigma = pd.read_csv(csv_path_fix_sigma)
print(f"Loaded {len(df_fix_sigma)} rows from {csv_path_fix_sigma}")
df_fix_sigma.head()

In [ ]:
required = {"dims", "mu_init", "sigma_init"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

available_dims = sorted(df["dims"].unique())
print("Available dims:", available_dims)

In [ ]:
beta_steps_prior = df["beta_steps_prior_mean"].iloc[0]
like_evals_prior = df["like_evals_prior_mean"].iloc[0]

In [ ]:
# Pick one dims value to visualize
dims_to_plot = available_dims[0]  # change this if needed

sub = df[df["dims"] == dims_to_plot].copy()

# If there are repeated (mu, sigma) points, average numeric summary columns
value_cols = [
    c
    for c in ["beta_steps_mean", "beta_steps_std", "like_evals_mean", "like_evals_std"]
    if c in sub.columns
]

if value_cols:
    agg = {c: "mean" for c in value_cols}
    grid_df = sub.groupby(["mu_init", "sigma_init"], as_index=False).agg(agg)
else:
    grid_df = sub[["mu_init", "sigma_init"]].drop_duplicates().copy()

grid_df = grid_df.sort_values(["mu_init", "sigma_init"]).reset_index(drop=True)
print(f"dims={dims_to_plot}: {len(grid_df)} unique (mu, sigma) points")
grid_df.head()

In [ ]:
beta_steps_prior

## Hellinger Distances

Compute analytic Hellinger distances for isotropic Gaussian pairs:
- \(H(q_{init}, p(x\mid y))\) for each proposal grid point
- \(H(\text{prior}, p(x\mid y))\) as a baseline scalar

In [ ]:
# Set these to the values used when generating the CSV
mu_prior = 0.0
sigma_prior = 3.0
y_obs = 2.0
sigma_obs = 1.0


def posterior_parameters(mu_prior, sigma_prior, y_obs, sigma_obs):
    var_prior = sigma_prior**2
    var_obs = sigma_obs**2
    var_post = 1.0 / (1.0 / var_prior + 1.0 / var_obs)
    mu_post = var_post * (mu_prior / var_prior + y_obs / var_obs)
    return mu_post, np.sqrt(var_post)


mu_post, sigma_post = posterior_parameters(mu_prior, sigma_prior, y_obs, sigma_obs)
print(f"Posterior params (per-dim): mu={mu_post:.6g}, sigma={sigma_post:.6g}")

In [ ]:
def gaussian_kl(mu_q, sigma_q, mu_p, sigma_p):
    var_q = sigma_q**2
    var_p = sigma_p**2
    return 0.5 * (np.log(var_p / var_q) + (var_q + (mu_q - mu_p) ** 2) / var_p - 1.0)


def total_kl_to_posterior(mu_q, sigma_q, mu_post, sigma_post, dims):
    return dims * gaussian_kl(mu_q, sigma_q, mu_post, sigma_post)


def total_kl_posterior_to_q(mu_q, sigma_q, mu_post, sigma_post, dims):
    return dims * gaussian_kl(mu_post, sigma_post, mu_q, sigma_q)


def total_wasserstein_distance(mu_q, sigma_q, mu_ref, sigma_ref, dims):
    delta_mu = mu_q - mu_ref
    delta_sigma = sigma_q - sigma_ref
    return np.sqrt(dims * (delta_mu**2 + delta_sigma**2))


def hellinger_isotropic_gaussians(mu1, sigma1, mu2, sigma2, dims):
    # Bhattacharyya coefficient for one dimension
    bc1 = np.sqrt((2.0 * sigma1 * sigma2) / (sigma1**2 + sigma2**2))
    bc1 *= np.exp(-((mu1 - mu2) ** 2) / (4.0 * (sigma1**2 + sigma2**2)))
    bc1 = np.clip(bc1, 0.0, 1.0)
    bcd = bc1**dims
    return np.sqrt(np.clip(1.0 - bcd, 0.0, 1.0))


grid_df["kl_init_posterior"] = total_kl_to_posterior(
    mu_q=grid_df["mu_init"].to_numpy(),
    sigma_q=grid_df["sigma_init"].to_numpy(),
    mu_post=mu_post,
    sigma_post=sigma_post,
    dims=dims_to_plot,
)

grid_df["kl_posterior_init"] = total_kl_posterior_to_q(
    mu_q=grid_df["mu_init"].to_numpy(),
    sigma_q=grid_df["sigma_init"].to_numpy(),
    mu_post=mu_post,
    sigma_post=sigma_post,
    dims=dims_to_plot,
)

grid_df["w2_init_posterior"] = total_wasserstein_distance(
    mu_q=grid_df["mu_init"].to_numpy(),
    sigma_q=grid_df["sigma_init"].to_numpy(),
    mu_ref=mu_post,
    sigma_ref=sigma_post,
    dims=dims_to_plot,
)

grid_df["hellinger_init_posterior"] = hellinger_isotropic_gaussians(
    mu1=grid_df["mu_init"].to_numpy(),
    sigma1=grid_df["sigma_init"].to_numpy(),
    mu2=mu_post,
    sigma2=sigma_post,
    dims=dims_to_plot,
)

kl_prior_posterior = total_kl_to_posterior(
    mu_prior, sigma_prior, mu_post, sigma_post, dims_to_plot
)
kl_posterior_prior = total_kl_posterior_to_q(
    mu_prior, sigma_prior, mu_post, sigma_post, dims_to_plot
)
w2_prior_posterior = total_wasserstein_distance(
    mu_prior, sigma_prior, mu_post, sigma_post, dims_to_plot
)
hellinger_prior_posterior = hellinger_isotropic_gaussians(
    mu1=mu_prior, sigma1=sigma_prior, mu2=mu_post, sigma2=sigma_post, dims=dims_to_plot
)

print(f"KL(prior||posterior) for dims={dims_to_plot}: {kl_prior_posterior:.6g}")
print(f"KL(posterior||prior) for dims={dims_to_plot}: {kl_posterior_prior:.6g}")
print(f"W2(prior, posterior) for dims={dims_to_plot}: {w2_prior_posterior:.6g}")
print(f"H(prior, posterior) for dims={dims_to_plot}: {hellinger_prior_posterior:.6g}")
grid_df[
    [
        "mu_init",
        "sigma_init",
        "kl_init_posterior",
        "kl_posterior_init",
        "w2_init_posterior",
        "hellinger_init_posterior",
    ]
].head()

In [ ]:
# Plot slice of KL divergence to posterior across (mu, sigma) grid
# Plot different mus for sigma close to posterior sigma

import cmcrameri.cm as cmc
from matplotlib import colors

plt.rcParams["lines.linewidth"] = 1.2

prior_ref_line_color = "tab:cyan"
prior_ref_line_style = "-"

mu_slice_color = "tab:orange"
sigma_slice_color = "tab:red"

mu_slice_lstyle = "--"
sigma_slice_lstyle = "-."

shading_color = "lightgrey"

figsize = copy.copy(plt.rcParams.get("figure.figsize"))
figsize[0] = 2 * figsize[0]  # make wider to fit 3 subplots
figsize[1] = 3 * figsize[1]  # make taller to fit 2 rows of subplots
fig, axs = plt.subplots(3, 3, figsize=figsize)

cbar_pad = (
    0.02  # adjust this value to increase/decrease space between subplots and colorbars
)
cbar_location = "top"
cmap = cmc.lipari

pivot = grid_df.pivot(index="mu_init", columns="sigma_init", values="kl_posterior_init")
pivot = pivot.sort_index(axis=0).sort_index(axis=1)

mu_vals = pivot.index.to_numpy()
sigma_vals = pivot.columns.to_numpy()
Z = pivot.to_numpy().T

if np.isnan(Z).any():
    print(
        "Warning: grid contains NaNs (missing mu/sigma combinations). Heatmap may show gaps."
    )

mu_vals[:5], sigma_vals[:5], Z.shape

pivot_h = grid_df.pivot(
    index="mu_init", columns="sigma_init", values="hellinger_init_posterior"
)
pivot_h = pivot_h.sort_index(axis=0).sort_index(axis=1)

mu_vals_h = pivot_h.index.to_numpy()
sigma_vals_h = pivot_h.columns.to_numpy()
H = pivot_h.to_numpy().T

pivot_evals = grid_df.pivot(
    index="mu_init", columns="sigma_init", values="like_evals_mean"
)
pivot_evals = pivot_evals.sort_index(axis=0).sort_index(axis=1)

mu_vals_h = pivot_evals.index.to_numpy()
sigma_vals_h = pivot_evals.columns.to_numpy()
like_evals = pivot_evals.to_numpy().T


norm = colors.LogNorm(
    vmin=max(Z.min(), 1e-12),  # avoid zero or negative values for log scale
    vmax=Z.max(),
)

im = axs[0, 0].imshow(
    Z,
    origin="lower",
    aspect="auto",
    extent=[mu_vals.min(), mu_vals.max(), sigma_vals.min(), sigma_vals.max()],
    interpolation="nearest",
    cmap=cmap,
    # vmax=10,
    norm=norm,
)

cs = axs[0, 0].contour(
    mu_vals,
    sigma_vals,
    Z,
    levels=[kl_posterior_prior],
    colors=prior_ref_line_color,
    linestyles=prior_ref_line_style,
    linewidths=1.2,
    alpha=0.8,
)

cbar = fig.colorbar(im, ax=axs[0, 0], pad=cbar_pad, location=cbar_location)
cbar.set_label(r"$D_{\rm KL}\;[{\rm nats}]$")  # [p(\theta|d, M)||q(\theta)]$')
cbar.ax.yaxis.set_label_position("left")
cbar.ax.axvline(
    kl_posterior_prior,
    color=prior_ref_line_color,
    linestyle=prior_ref_line_style,
    linewidth=1.2,
    alpha=0.8,
)

norm = colors.LogNorm(
    vmin=kl_posterior_prior,  # avoid zero or negative values for log scale
    vmax=Z.max(),
)

norm = colors.LogNorm(
    vmin=max(H.min(), 1e-12),  # avoid zero or negative values for log scale
    vmax=1,
)


im = axs[0, 1].imshow(
    H,
    origin="lower",
    aspect="auto",
    # extent=[sigma_vals_h.min(), sigma_vals_h.max(), mu_vals_h.min(), mu_vals_h.max()],
    extent=[mu_vals_h.min(), mu_vals_h.max(), sigma_vals_h.min(), sigma_vals_h.max()],
    interpolation="nearest",
    cmap=cmap,
)

# Add contour for prior Hellinger distance
cs = axs[0, 1].contour(
    mu_vals_h,
    sigma_vals_h,
    H,
    levels=[hellinger_prior_posterior],
    colors=prior_ref_line_color,
    linestyles=prior_ref_line_style,
    linewidths=1.2,
    alpha=0.8,
)

cbar = fig.colorbar(im, ax=axs[0, 1], pad=cbar_pad, location=cbar_location)
cbar.set_label(r"$D_{{\rm H}^2}$")  # [p(\theta|d, M)||q(\theta)]$')
cbar.ax.yaxis.set_label_position("left")
cbar.ax.axvline(
    hellinger_prior_posterior,
    color=prior_ref_line_color,
    linestyle=prior_ref_line_style,
    linewidth=1.2,
    alpha=0.8,
)


im = axs[0, 2].imshow(
    like_evals,
    origin="lower",
    aspect="auto",
    extent=[mu_vals_h.min(), mu_vals_h.max(), sigma_vals_h.min(), sigma_vals_h.max()],
    interpolation="nearest",
    cmap=cmap,
    # norm=norm
)
cs = axs[0, 2].contour(
    mu_vals,
    sigma_vals,
    like_evals,
    levels=[like_evals_prior],
    colors=prior_ref_line_color,
    linestyles=prior_ref_line_style,
    linewidths=1.2,
    alpha=0.8,
)
# Place colorbar above the subplot
cbar = fig.colorbar(im, ax=axs[0, 2], pad=cbar_pad, location=cbar_location)
cbar.set_label("Likelihood evaluations")
cbar.ax.yaxis.set_label_position("left")
cbar.ax.axvline(
    like_evals_prior,
    color=prior_ref_line_color,
    linestyle=prior_ref_line_style,
    linewidth=1.2,
    alpha=0.8,
)

# Plot posterior mean and standard deviation as lines
for ax in axs[0]:
    ax.axvline(
        mu_post,
        color=sigma_slice_color,
        linestyle=sigma_slice_lstyle,
        linewidth=1.2,
        alpha=0.8,
        label="posterior mean",
    )
    ax.axhline(
        sigma_post,
        color=mu_slice_color,
        linestyle=mu_slice_lstyle,
        linewidth=1.2,
        alpha=0.8,
        label="posterior sigma",
    )
    # ax.scatter(mu_post, sigma_post, color="white", marker='^', label='posterior mean/sigma')
    # ax.scatter(mu_prior, sigma_prior, color="white", marker='d', label='prior mean/sigma')


for ax in axs[0]:
    ax.set_xlabel("$\mu_{q}$")
    ax.set_ylabel("$\sigma_{q}$")


print(f"True posterior mean: {mu_post:.6g}, posterior sigma: {sigma_post:.6g}")

mu_vals = df_fix_sigma["mu_init"].unique()
sigma_vals = df_fix_sigma["sigma_init"].unique()
print(f"Unique mu values in fix_sigma data: {mu_vals}")
print(f"Unique sigma values in fix_sigma data: {sigma_vals}")

# Compute KL and Hellinger values for the fix_sigma data using the functions defined above
kl_vals = total_kl_to_posterior(
    mu_q=df_fix_sigma["mu_init"].to_numpy(),
    sigma_q=df_fix_sigma["sigma_init"].to_numpy(),
    mu_post=mu_post,
    sigma_post=sigma_post,
    dims=dims_to_plot,
)
hell_vals = hellinger_isotropic_gaussians(
    mu1=df_fix_sigma["mu_init"].to_numpy(),
    sigma1=df_fix_sigma["sigma_init"].to_numpy(),
    mu2=mu_post,
    sigma2=sigma_post,
    dims=dims_to_plot,
)
evals = df_fix_sigma["like_evals_mean"].to_numpy()
mu_interp = np.interp(like_evals_prior, evals, mu_vals)
mu_grid = np.linspace(mu_vals.min(), mu_vals.max(), 100)
evals_interp = np.interp(mu_grid, mu_vals, evals)
like_evals_prior_interp = np.interp(mu_grid, mu_vals, evals)

axs[1, 0].plot(
    mu_vals,
    kl_vals,
    color=mu_slice_color,
    linestyle=mu_slice_lstyle,
)
axs[1, 0].axhline(
    kl_posterior_prior,
    color=prior_ref_line_color,
    linestyle=prior_ref_line_style,
    linewidth=1.2,
    alpha=0.8,
)
axs[1, 0].fill_between(
    mu_grid,
    kl_vals.min(),
    kl_posterior_prior,
    where=(evals_interp < like_evals_prior),
    color=shading_color,
    alpha=0.3,
    interpolate=True,
)

axs[1, 1].plot(
    mu_vals,
    hell_vals,
    color=mu_slice_color,
    linestyle=mu_slice_lstyle,
)
axs[1, 1].axhline(
    hellinger_prior_posterior,
    color=prior_ref_line_color,
    linestyle=prior_ref_line_style,
    linewidth=1.2,
    alpha=0.8,
)
# Shade corresponding area in Hellinger plot
axs[1, 1].fill_between(
    mu_grid,
    hell_vals.min(),
    hellinger_prior_posterior,
    where=(evals_interp < like_evals_prior),
    color=shading_color,
    alpha=0.3,
    interpolate=True,
)

axs[1, 2].plot(
    mu_vals,
    evals,
    color=mu_slice_color,
    linestyle=mu_slice_lstyle,
)
axs[1, 2].axhline(
    like_evals_prior,
    color=prior_ref_line_color,
    linestyle=prior_ref_line_style,
    linewidth=1.2,
    alpha=0.8,
)
# Shade rectanglea where evals are below the prior reference line
# Estimate evals at exact the prior line
axs[1, 2].fill_between(
    mu_grid,
    evals.min(),
    like_evals_prior,
    where=(evals_interp < like_evals_prior),
    color=shading_color,
    alpha=0.3,
    interpolate=True,
)

mu_vals = df_fix_mu["mu_init"].unique()
sigma_vals = df_fix_mu["sigma_init"].unique()
print(f"Unique sigma values in fix_mu data: {sigma_vals}")

kl_vals = total_kl_to_posterior(
    mu_q=df_fix_mu["mu_init"].to_numpy(),
    sigma_q=df_fix_mu["sigma_init"].to_numpy(),
    mu_post=mu_post,
    sigma_post=sigma_post,
    dims=dims_to_plot,
)
hell_vals = hellinger_isotropic_gaussians(
    mu1=df_fix_mu["mu_init"].to_numpy(),
    sigma1=df_fix_mu["sigma_init"].to_numpy(),
    mu2=mu_post,
    sigma2=sigma_post,
    dims=dims_to_plot,
)
evals = df_fix_mu["like_evals_mean"].to_numpy()

# Interpolate evals at the prior reference line for shading
sigma_interp = np.interp(like_evals_prior, evals, sigma_vals)
sigma_grid = np.linspace(sigma_vals.min(), sigma_vals.max(), 100)
evals_interp = np.interp(sigma_grid, sigma_vals, evals)
like_evals_prior_interp = np.interp(sigma_grid, sigma_vals, evals)

axs[2, 0].plot(
    sigma_vals,
    kl_vals,
    color=sigma_slice_color,
    linestyle=sigma_slice_lstyle,
)
axs[2, 0].axhline(
    kl_prior_posterior,
    color=prior_ref_line_color,
    linestyle=prior_ref_line_style,
    linewidth=1.2,
    alpha=0.8,
)
axs[2, 0].fill_between(
    sigma_grid,
    0,
    kl_prior_posterior,
    where=(evals_interp < like_evals_prior),
    color=shading_color,
    alpha=0.3,
    interpolate=True,
)

axs[2, 1].plot(
    sigma_vals,
    hell_vals,
    color=sigma_slice_color,
    linestyle=sigma_slice_lstyle,
)
axs[2, 1].axhline(
    hellinger_prior_posterior,
    color=prior_ref_line_color,
    linestyle=prior_ref_line_style,
    linewidth=1.2,
    alpha=0.8,
)
axs[2, 1].fill_between(
    sigma_grid,
    0,
    hellinger_prior_posterior,
    where=(evals_interp < like_evals_prior),
    color=shading_color,
    alpha=0.3,
    interpolate=True,
)

axs[2, 2].plot(
    sigma_vals,
    evals,
    color=sigma_slice_color,
    linestyle=sigma_slice_lstyle,
)
axs[2, 2].axhline(
    like_evals_prior,
    color=prior_ref_line_color,
    linestyle=prior_ref_line_style,
    linewidth=1.2,
    alpha=0.8,
)
axs[2, 2].fill_between(
    sigma_grid,
    evals.min(),
    like_evals_prior,
    where=(evals_interp < like_evals_prior),
    color=shading_color,
    alpha=0.3,
    interpolate=True,
)

for ax in axs[1]:
    ax.set_xlabel("$\mu_q$")
    ax.set_xlim(left=df_fix_sigma["mu_init"].min(), right=df_fix_sigma["mu_init"].max())
    ax.axvline(
        mu_post,
        color=sigma_slice_color,
        linestyle=sigma_slice_lstyle,
        linewidth=1.2,
        alpha=0.8,
    )

for ax in axs[2]:
    ax.set_xlabel("$\sigma_q$")
    ax.set_xlim(left=df_fix_mu["sigma_init"].min(), right=df_fix_mu["sigma_init"].max())
    ax.axvline(
        sigma_post,
        color=mu_slice_color,
        linestyle=mu_slice_lstyle,
        linewidth=1.2,
        alpha=0.8,
    )

for ax in axs[1:, 0]:
    ax.set_ylabel(r"$D_{\rm KL}$")
    ax.set_ylim(bottom=0)  # KL divergence cannot be negative
for ax in axs[1:, 1]:
    ax.set_ylabel(r"$D_{{\rm H}^2}$")
    ax.set_ylim(bottom=0, top=1)  # Hellinger distance cannot be negative
for ax in axs[1:, 2]:
    ax.set_ylabel("Likelihood evaluations")
    ax.set_ylim(evals.min(), evals.max())

for ax in axs.flatten():
    ax.grid()


plt.tight_layout()
fig.savefig("distance_vs_proposal.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# Plot of just the boundary lines for the prior reference values

mu_vals = pivot.index.to_numpy()
sigma_vals = pivot.columns.to_numpy()
Z = pivot.to_numpy().T

if np.isnan(Z).any():
    print(
        "Warning: grid contains NaNs (missing mu/sigma combinations). Heatmap may show gaps."
    )

mu_vals[:5], sigma_vals[:5], Z.shape

pivot_h = grid_df.pivot(
    index="mu_init", columns="sigma_init", values="hellinger_init_posterior"
)
pivot_h = pivot_h.sort_index(axis=0).sort_index(axis=1)

mu_vals_h = pivot_h.index.to_numpy()
sigma_vals_h = pivot_h.columns.to_numpy()
H = pivot_h.to_numpy().T

pivot_evals = grid_df.pivot(
    index="mu_init", columns="sigma_init", values="like_evals_mean"
)
pivot_evals = pivot_evals.sort_index(axis=0).sort_index(axis=1)

mu_vals_h = pivot_evals.index.to_numpy()
sigma_vals_h = pivot_evals.columns.to_numpy()
like_evals = pivot_evals.to_numpy().T

fig, ax = plt.subplots()

cs = ax.contour(
    mu_vals,
    sigma_vals,
    like_evals,
    levels=[like_evals_prior],
    colors="k",
    linestyles="-",
    linewidths=1.2,
    alpha=0.8,
)


cs = ax.contour(
    mu_vals_h,
    sigma_vals_h,
    H,
    levels=[hellinger_prior_posterior],
    colors="C0",
    linestyles="--",
    linewidths=1.2,
    alpha=0.8,
)

cs = ax.contour(
    mu_vals,
    sigma_vals,
    Z,
    levels=[kl_posterior_prior],
    colors="C1",
    linestyles=":",
    linewidths=1.2,
    alpha=0.8,
)

ax.set_xlabel("$\mu_q$")
ax.set_ylabel("$\sigma_q$")

ax.legend(
    handles=[
        Line2D(
            [0],
            [0],
            color="k",
            linestyle="-",
            linewidth=1.2,
            alpha=0.8,
            label="Likelihood evaluations",
        ),
        Line2D(
            [0],
            [0],
            color="C0",
            linestyle="--",
            linewidth=1.2,
            alpha=0.8,
            label=r"$D_{{\rm H}^2}$",
        ),
        Line2D(
            [0],
            [0],
            color="C1",
            linestyle=":",
            linewidth=1.2,
            alpha=0.8,
            label=r"$D_{\rm KL}$",
        ),
    ],
    loc="upper left",
)

ax.grid()
fig.savefig("threshold_boundaries.pdf", bbox_inches="tight")